# 经典系统设计 — 高级数据工程师面试精讲

本笔记覆盖高频数据系统设计题，每题使用统一的结构化模板作答。

| 主题 | 出现频率 |
|------|----------|
| 设计日志聚合系统（Clickstream） | 高频 |
| 设计实时 Dashboard（Lambda/Kappa） | 高频 |
| 设计 Feature Store | 重要 |
| 设计 ETL 监控告警系统 | 高频 |
| 设计 Multi-tenant 数仓隔离 | 重要 |

### 系统设计回答模板

每道题按以下 6 步作答：

1. **需求澄清**：规模、延迟、正确性要求
2. **高层架构**：ASCII 图展示数据流
3. **关键组件设计**：技术选型 + 设计决策
4. **数据模型**：Schema 设计
5. **规模估算**：QPS / 存储 / 带宽
6. **权衡讨论**：优缺点 + 备选方案

---
## 1. 设计日志聚合系统（Clickstream）

### 1.1 需求澄清

**面试官问**："设计一个处理用户点击流数据的日志聚合系统"

**你要问**：
- 每秒事件量？（假设：100 万 QPS）
- 延迟要求？（假设：分析延迟 T+1 可接受，实时告警 < 5min）
- 数据保留时长？（假设：原始日志 90 天，聚合数据 3 年）
- 需要支持 Session 重建吗？（假设：是，Session 超时 30 分钟）
- 需要精确去重吗？（假设：近似即可，误差 < 1%）

### 1.2 高层架构

```
客户端 (Web/App/SDK)
       │  HTTP POST / WebSocket
       ▼
  ┌─────────────┐
  │  API Gateway │  ← 限流 / 认证 / 序列化
  │  (Nginx/ALB) │
  └──────┬───────┘
         │  高吞吐写入
         ▼
  ┌─────────────┐
  │    Kafka     │  ← 消息队列缓冲 (多 partition，按 user_id 分区)
  │  (raw topic) │
  └──────┬───────┘
         │
    ┌────┴────────────────────┐
    │                         │
    ▼                         ▼
┌──────────┐          ┌──────────────┐
│  Flink   │          │   S3/HDFS    │
│ Streaming│          │  (raw sink)  │
│ (流处理)  │          │  Parquet     │
└────┬─────┘          └──────┬───────┘
     │ 实时聚合/Session重建        │ 批处理
     ▼                         ▼
┌──────────┐          ┌──────────────┐
│  Redis   │          │  Spark/dbt   │
│(session  │          │  (T+1 ETL)   │
│ state)   │          └──────┬───────┘
└──────────┘                 │
                             ▼
                    ┌──────────────┐
                    │  Hive/Trino  │
                    │  (DWH mart)  │
                    │  BI / Reports│
                    └──────────────┘
```

### 1.3 关键组件设计

**Kafka 分区策略**：
- 按 `user_id % N` 分区，保证同一用户事件有序到达同一 partition
- Session 重建不需要跨 partition 协调
- Partition 数 = max_consumers × 2（预留扩容空间）

**去重策略**：
- 客户端生成 `event_id = UUID`，Flink 使用 `RocksDB` 状态存储已见 event_id（TTL 24h）
- 近似去重：HyperLogLog（误差 ~1.6%，内存 O(1)）

**乱序事件处理**：
- Flink Watermark：允许 5 分钟延迟，`event_time - 5min` 作为 watermark
- 超过 watermark 的晚到数据写入 `late_data_topic` 单独处理

**Session 重建**：
- Flink Session Window：`withGap(30 minutes)` 基于 event_time 分组
- Redis 存储 `{user_id: last_event_time}`，Flink 查询判断是否超时

### 1.4 数据模型

```sql
-- Raw event (Kafka message schema / Parquet schema)
CREATE TABLE raw_clickstream (
    event_id      STRING,       -- UUID, for dedup
    event_time    TIMESTAMP,    -- client-side timestamp
    ingest_time   TIMESTAMP,    -- server-side arrival time
    user_id       STRING,
    session_id    STRING,       -- client-generated session ID
    event_type    STRING,       -- 'page_view', 'click', 'purchase'
    page_url      STRING,
    element_id    STRING,
    properties    MAP<STRING,STRING>,  -- flexible attributes
    device_type   STRING,
    ip_address    STRING
)
PARTITIONED BY (dt STRING, hr STRING);  -- date/hour partitions

-- Aggregated: hourly page views by URL
CREATE TABLE agg_hourly_pageviews (
    dt            DATE,
    hr            INTEGER,
    page_url      STRING,
    pv_count      BIGINT,
    uv_count      BIGINT,   -- HyperLogLog estimate
    avg_dwell_sec DOUBLE
);
```

### 1.5 规模估算

| 指标 | 估算 |
|------|------|
| 峰值 QPS | 100 万 events/s |
| 每事件大小 | ~500 bytes (JSON) |
| 原始数据带宽 | 500 MB/s = 1.8 TB/hr |
| 日原始存储 (Parquet 压缩 ~10x) | ~4 TB/day |
| Kafka 需要 | ~50 partition，retention 24h ≈ 40 TB |
| Flink 节点 | 20 TaskManager × 8 core |

### 1.6 权衡讨论

| 决策 | 选择 | 备选 | 理由 |
|------|------|------|------|
| 消息队列 | Kafka | Pulsar, Kinesis | 成熟生态，高吞吐 |
| 流处理 | Flink | Spark Streaming | Flink 事件时间处理更精确 |
| 存储格式 | Parquet | ORC, Avro | 列式压缩，查询友好 |
| Session 状态 | Redis | Flink State | Redis 可跨 job 共享 |

In [ ]:
# Simulate clickstream processing: dedup + session reconstruction
import pandas as pd
import duckdb
from datetime import datetime, timedelta
import uuid
import random

random.seed(42)

# Generate synthetic clickstream events (with duplicates and out-of-order)
def generate_events(n_users=5, events_per_user=8):
    events = []
    base_time = datetime(2024, 1, 1, 10, 0, 0)
    pages = ["/home", "/product", "/cart", "/checkout", "/search"]

    for user_num in range(n_users):
        user_id = f"U{user_num+1:03d}"
        t = base_time + timedelta(minutes=random.randint(0, 30))
        for i in range(events_per_user):
            event_id = str(uuid.uuid4())[:8]
            event = {
                "event_id":   event_id,
                "event_time": t,
                "user_id":    user_id,
                "event_type": random.choice(["page_view", "click", "page_view"]),
                "page_url":   random.choice(pages),
            }
            events.append(event)
            # Introduce duplicates (~20% of events)
            if random.random() < 0.2:
                events.append(event.copy())  # exact duplicate
            t += timedelta(seconds=random.randint(10, 200))

    # Shuffle to simulate out-of-order arrival
    random.shuffle(events)
    return pd.DataFrame(events)

raw_events = generate_events()
print(f"Raw events (with duplicates): {len(raw_events)} rows")

con = duckdb.connect(":memory:")
con.register("raw_events", raw_events)

# Step 1: Deduplication using event_id
deduped = con.execute("""
    SELECT DISTINCT ON (event_id) *
    FROM raw_events
    ORDER BY event_id, event_time
""").fetchdf()
print(f"After dedup:                  {len(deduped)} rows")
print(f"Duplicates removed:           {len(raw_events) - len(deduped)}")
con.register("deduped_events", deduped)

# Step 2: Session reconstruction (30-min gap = new session)
sessions = con.execute("""
    WITH ordered AS (
        SELECT *,
            LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) AS prev_time
        FROM deduped_events
    ),
    session_flags AS (
        SELECT *,
            CASE
                WHEN prev_time IS NULL
                  OR DATEDIFF('minute', prev_time, event_time) > 30
                THEN 1 ELSE 0
            END AS new_session_flag
        FROM ordered
    ),
    with_session AS (
        SELECT *,
            user_id || '_S' || LPAD(
                CAST(SUM(new_session_flag) OVER (
                    PARTITION BY user_id ORDER BY event_time
                ) AS VARCHAR), 2, '0'
            ) AS session_id
        FROM session_flags
    )
    SELECT
        session_id,
        user_id,
        MIN(event_time)             AS session_start,
        MAX(event_time)             AS session_end,
        COUNT(*)                    AS event_count,
        DATEDIFF('second',
            MIN(event_time), MAX(event_time)) AS duration_sec,
        COUNT(*) FILTER (WHERE event_type = 'page_view') AS pageviews
    FROM with_session
    GROUP BY session_id, user_id
    ORDER BY user_id, session_start
""").fetchdf()

print("\n=== Reconstructed Sessions ===")
print(sessions.to_string(index=False))

---
## 2. 设计实时 Dashboard（Lambda / Kappa 架构）

### 2.1 需求澄清

- 延迟要求：< 5 秒展示最新数据（准实时）
- 历史查询：最近 90 天任意时间段
- 并发用户：500 人同时刷新
- 数据来源：Kafka 实时流 + S3 历史存量

### 2.2 Lambda 架构

```
数据源 (Kafka)
    │
    ├──────────────────────────────────┐
    │                                  │
    ▼  批处理层 (Batch Layer)           ▼  速度层 (Speed Layer)
┌────────────┐                    ┌────────────┐
│  Spark     │                    │  Flink /   │
│  (T+1 ETL) │                    │  Spark     │
│  精确计算   │                    │  Streaming │
└─────┬──────┘                    └─────┬──────┘
      │                                 │
      ▼ 历史精确数据                     ▼ 近实时增量数据
┌─────────────────────────────────────────────┐
│           服务层 (Serving Layer)             │
│   Druid / Apache Pinot / Redis               │
│   合并 Batch + Speed 结果响应查询            │
└─────────────────────┬───────────────────────┘
                      │
                      ▼
               Dashboard (Grafana / Superset)
```

**Lambda 优缺点**：
- 优：批处理保证精确性，速度层保证低延迟；容错性好
- 缺：维护两套代码（批 + 流），逻辑同步难，运维复杂

### 2.3 Kappa 架构

```
数据源 (Kafka — 保留足够长时间，如 90 天)
    │
    ▼
┌──────────────────────────────────────────┐
│         流处理层 (Streaming Only)         │
│   Flink / Spark Streaming                │
│   · 实时处理当前数据                      │
│   · 历史重算：从头回放 Kafka topic        │
│   · 新旧版本并行跑，切换后下线旧版本      │
└──────────────────┬───────────────────────┘
                   │
                   ▼
          ┌────────────────┐
          │  Serving Store │
          │  Pinot / Druid │
          └────────────────┘
                   │
                   ▼
            Dashboard
```

**Kappa 优缺点**：
- 优：单一代码路径，维护简单；历史重算 = 重放 Kafka
- 缺：依赖 Kafka 长期保留（成本高）；批处理类型的任务（全量 JOIN）不适合

### 2.4 何时选择 Lambda vs Kappa？

| 场景 | Lambda | Kappa |
|------|--------|-------|
| 需要精确批处理（如月度对账） | ✓ | ✗ |
| 逻辑简单，只需聚合 | ✗ | ✓ |
| 历史数据量超大（PB+） | ✓（批处理更经济） | ✗（Kafka 成本） |
| 团队小，希望简单 | ✗ | ✓ |
| 需要复杂历史 JOIN | ✓ | ✗ |

In [ ]:
# Simulate Lambda architecture: merge batch (precise) + speed (recent) results
import pandas as pd
import duckdb
from datetime import datetime, timedelta

con = duckdb.connect(":memory:")

# Batch layer result: precise daily aggregation (T+1, covers up to yesterday)
batch_data = pd.DataFrame([
    {"date": "2024-01-01", "page": "/home",    "pv": 125000, "uv": 45000, "source": "batch"},
    {"date": "2024-01-01", "page": "/product", "pv":  80000, "uv": 32000, "source": "batch"},
    {"date": "2024-01-02", "page": "/home",    "pv": 131000, "uv": 47000, "source": "batch"},
    {"date": "2024-01-02", "page": "/product", "pv":  85000, "uv": 34000, "source": "batch"},
])

# Speed layer result: real-time partial aggregation for today (last 5 min window)
speed_data = pd.DataFrame([
    {"date": "2024-01-03", "page": "/home",    "pv": 12000,  "uv": 5200,  "source": "speed"},
    {"date": "2024-01-03", "page": "/product", "pv":  7800,  "uv": 3100,  "source": "speed"},
])

con.register("batch_layer", batch_data)
con.register("speed_layer", speed_data)

# Serving layer: UNION batch + speed, query covers all dates
serving_result = con.execute("""
    -- Serving layer merges batch (precise historical) + speed (recent approximate)
    SELECT date, page, SUM(pv) AS total_pv, SUM(uv) AS total_uv,
           STRING_AGG(source, '+') AS data_sources
    FROM (
        SELECT * FROM batch_layer
        UNION ALL
        SELECT * FROM speed_layer
    ) combined
    GROUP BY date, page
    ORDER BY date, page
""").fetchdf()

print("=== Lambda Architecture: Serving Layer (Batch + Speed merged) ===")
print(serving_result.to_string(index=False))

print("\n=== Architecture Comparison ===")
comparison = pd.DataFrame([
    {"Architecture": "Lambda", "Complexity": "High (2 codebases)", "Latency": "<5s",
     "Accuracy": "Precise (batch) + Approx (speed)", "Best for": "Complex batch + real-time"},
    {"Architecture": "Kappa",  "Complexity": "Low (1 codebase)",  "Latency": "<5s",
     "Accuracy": "Approximate", "Best for": "Simple streaming aggregations"},
])
print(comparison.to_string(index=False))

---
## 3. 设计 Feature Store

### 3.1 需求澄清

- 特征数量：~5000 个特征，每天新增 ~20 个
- 在线推理延迟：< 10ms（p99）
- 离线训练数据：支持任意时间点的历史特征（Point-in-Time Correct）
- 团队：ML 团队自主注册和消费特征，数据工程团队管理基础设施

### 3.2 高层架构

```
原始数据 (S3 / DWH)
       │
       ▼
┌──────────────────────────────────────────────────┐
│              特征计算层 Feature Pipeline          │
│  Spark Batch Jobs (daily/hourly)                  │
│  Flink Streaming (real-time features)             │
└──────────┬───────────────────────┬────────────────┘
           │                       │
           ▼                       ▼
┌──────────────────┐    ┌──────────────────────────┐
│   离线存储        │    │      在线存储              │
│  Offline Store   │    │     Online Store          │
│                  │    │                           │
│  S3 + Parquet    │    │  Redis / DynamoDB         │
│  Hive / Delta    │    │  (低延迟 KV 查找)          │
│                  │    │                           │
│  用于：           │    │  用于：                    │
│  · 训练数据生成   │    │  · 实时推理服务            │
│  · 历史回填      │    │  · < 10ms 延迟要求         │
│  · 特征探索      │    │                           │
└──────────┬───────┘    └───────────────────────────┘
           │                       ▲
           ▼                       │ 物化 (Materialization)
┌──────────────────────────────────┴───────────────┐
│              特征注册表 Feature Registry           │
│  · 特征元数据（名称/类型/所有者/描述）              │
│  · 血缘追踪（特征 ← 数据源）                       │
│  · 版本管理                                        │
│  · 访问控制                                        │
└──────────────────────────────────────────────────┘
```

### 3.3 关键设计：Point-in-Time Correct Join

**问题**：训练数据中，特征值必须使用标签时间点 **之前** 的最新值，否则造成数据泄露（Data Leakage）。

```
事件时间轴：
  ──────────────────────────────────────────────────>
  t1        t2                    t3           t4
  特征更新   特征更新               标签发生      当前
  (v1)      (v2)                 (用 v2 训练)  (不能用 t4 的特征！)

Point-in-Time Correct JOIN 确保：在 t3 时刻，使用的是 t2 的特征值
```

### 3.4 数据模型

```sql
-- 离线特征表（每个特征组一张表，按时间分区）
CREATE TABLE feature_user_stats (
    entity_id        STRING,    -- user_id
    feature_timestamp TIMESTAMP, -- when this feature value was computed
    created_ts        TIMESTAMP, -- when this row was ingested
    -- feature values
    purchase_count_7d  INTEGER,
    avg_order_value    DOUBLE,
    days_since_login   INTEGER
)
PARTITIONED BY (dt DATE);

-- 特征注册表
CREATE TABLE feature_registry (
    feature_name       VARCHAR(200) PRIMARY KEY,
    feature_group      VARCHAR(100),
    owner              VARCHAR(100),
    description        TEXT,
    data_type          VARCHAR(50),
    source_table       VARCHAR(200),
    freshness_sla      INTERVAL,
    created_at         TIMESTAMP,
    is_online          BOOLEAN     -- whether materialized to online store
);
```

### 3.5 规模估算

| 指标 | 估算 |
|------|------|
| 特征数 | 5000 |
| 实体数（用户） | 1 亿 |
| 在线存储（Redis）| 5000 features × 1亿 users × 50 bytes = 25 TB |
| 在线查询 QPS | 100k/s（推理服务并发） |
| 离线训练数据 | 5000 features × 3年历史 = PB 级 |

### 3.6 Feast vs Tecton

| 维度 | Feast（开源） | Tecton（商业） |
|------|--------------|---------------|
| 成本 | 免费 | 贵（但功能完整） |
| PIT Join | 支持 | 支持（更自动化） |
| 在线存储 | Redis/DynamoDB 自管 | 托管 |
| 实时特征 | 有限 | 原生支持 |
| 适用场景 | 小团队/POC | 大型 ML 平台 |

In [ ]:
# Simulate Point-in-Time Correct Join for Feature Store
import pandas as pd
import duckdb

con = duckdb.connect(":memory:")

# Feature table: user statistics (updated daily)
feature_data = pd.DataFrame([
    # user U001 feature history
    {"user_id": "U001", "feature_ts": "2024-01-01", "purchase_count_7d": 3,  "avg_order": 120.0},
    {"user_id": "U001", "feature_ts": "2024-01-05", "purchase_count_7d": 5,  "avg_order": 135.0},
    {"user_id": "U001", "feature_ts": "2024-01-10", "purchase_count_7d": 8,  "avg_order": 142.0},
    # user U002 feature history
    {"user_id": "U002", "feature_ts": "2024-01-01", "purchase_count_7d": 1,  "avg_order":  80.0},
    {"user_id": "U002", "feature_ts": "2024-01-08", "purchase_count_7d": 2,  "avg_order":  90.0},
])

# Training labels: user behavior events with timestamps
# We want: features AS OF label_ts (not the latest features!)
labels = pd.DataFrame([
    {"user_id": "U001", "label_ts": "2024-01-07", "label": 1},  # should use features from 2024-01-05
    {"user_id": "U001", "label_ts": "2024-01-12", "label": 1},  # should use features from 2024-01-10
    {"user_id": "U002", "label_ts": "2024-01-05", "label": 0},  # should use features from 2024-01-01
    {"user_id": "U002", "label_ts": "2024-01-10", "label": 1},  # should use features from 2024-01-08
])

con.register("features", feature_data)
con.register("labels", labels)

# Point-in-Time Correct Join:
# For each label event, find the LATEST feature row where feature_ts <= label_ts
training_data = con.execute("""
    SELECT
        l.user_id,
        l.label_ts,
        l.label,
        f.feature_ts         AS feature_as_of,
        f.purchase_count_7d,
        f.avg_order
    FROM labels l
    JOIN features f ON l.user_id = f.user_id
        AND f.feature_ts <= l.label_ts   -- only use features BEFORE the label
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY l.user_id, l.label_ts
        ORDER BY f.feature_ts DESC       -- pick the most recent valid feature
    ) = 1
    ORDER BY l.user_id, l.label_ts
""").fetchdf()

print("=== Point-in-Time Correct Training Dataset ===")
print(training_data.to_string(index=False))

print("\n=== Verification: No Data Leakage ===")
for _, row in training_data.iterrows():
    assert row["feature_as_of"] <= row["label_ts"], "DATA LEAKAGE DETECTED!"
    print(f"  User {row['user_id']} @ label={row['label_ts']}: "
          f"used features from {row['feature_as_of']} ✓")

---
## 4. 设计 ETL 监控告警系统

### 4.1 需求澄清

- 监控对象：500 个 Airflow DAG，每天运行 2000+ 个任务
- 告警延迟：SLA 突破后 5 分钟内通知
- 告警渠道：PagerDuty（P1）、Slack（P2）、Email（P3）
- 误报率要求：< 5%（避免告警疲劳）

### 4.2 高层架构

```
Airflow / Spark / dbt
       │  metrics (duration, row_count, status)
       ▼
┌─────────────────────────────────────────────────────┐
│           指标采集层 Metrics Collection              │
│  · Airflow callbacks → Kafka → metrics DB           │
│  · Prometheus exporters (Spark metrics)             │
│  · dbt test results → metadata DB                  │
└──────────────────────┬──────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────┐
│         异常检测层 Anomaly Detection                 │
│  · Rule-based: duration > P95 * 1.5, row_count = 0  │
│  · Statistical: Z-score, IQR outlier detection      │
│  · ML-based: Prophet / LSTM for seasonality        │
│  · SLA 检查: 预计完成时间 vs 实际                    │
└──────────────────────┬──────────────────────────────┘
                       │ 告警事件
                       ▼
┌─────────────────────────────────────────────────────┐
│           告警路由层 Alert Router                    │
│  · 去重：同一 DAG 5 分钟内不重复告警               │
│  · 分级：P1→PagerDuty, P2→Slack, P3→Email         │
│  · 升级：30 分钟未处理自动升级 severity             │
│  · 静默：计划维护窗口期静默                         │
└──────────────────────┬──────────────────────────────┘
                       │
           ┌───────────┼───────────┐
           ▼           ▼           ▼
      PagerDuty      Slack       Email
```

### 4.3 关键组件：指标设计

```sql
-- 每次 ETL 任务运行的指标记录
CREATE TABLE etl_run_metrics (
    run_id           VARCHAR(100) PRIMARY KEY,
    dag_id           VARCHAR(200) NOT NULL,
    task_id          VARCHAR(200),
    run_date         DATE NOT NULL,
    scheduled_start  TIMESTAMP,
    actual_start     TIMESTAMP,
    actual_end       TIMESTAMP,
    duration_sec     INTEGER,
    status           VARCHAR(20),    -- 'success'/'failed'/'running'
    rows_read        BIGINT,
    rows_written     BIGINT,
    rows_rejected    BIGINT,
    error_message    TEXT,
    created_at       TIMESTAMP DEFAULT NOW()
);

-- SLA 定义
CREATE TABLE etl_sla (
    dag_id           VARCHAR(200) PRIMARY KEY,
    expected_duration_p95   INTEGER,   -- seconds, from historical P95
    max_allowed_duration    INTEGER,   -- hard limit
    sla_deadline     TIME,             -- must finish by
    min_rows_expected BIGINT,
    owner_email      VARCHAR(200),
    slack_channel    VARCHAR(100),
    priority         VARCHAR(5)        -- P1/P2/P3
);
```

### 4.4 异常检测策略

| 检测类型 | 方法 | 适用场景 |
|----------|------|----------|
| 基于规则 | `duration > P95 × 1.5` | 简单、可解释 |
| Z-score | `(x - mean) / std > 3` | 正态分布指标 |
| IQR | `x > Q3 + 1.5×IQR` | 非对称分布 |
| 时序预测 | Prophet / LSTM | 有周期性的指标（周/月规律） |
| 行数对比 | `abs(rows - expected) / expected > 20%` | 数据量突变 |

### 4.5 自动修复（Auto-Remediation）

| 故障类型 | 自动修复 |
|----------|----------|
| 任务超时 | 自动重试（backoff：1min/5min/15min） |
| 上游依赖未就绪 | 等待 + 重调度 |
| 资源不足（OOM）| 自动提升 executor 内存，重试 |
| 数据质量失败 | 发告警，**不** 自动修复（需人工判断） |

In [ ]:
# Implement ETL anomaly detection: Z-score + IQR on job duration
import pandas as pd
import numpy as np
import duckdb

np.random.seed(42)

# Simulate 30 days of historical ETL run durations (seconds)
# Normal jobs: ~300s, some anomalies injected
historical_durations = np.random.normal(loc=300, scale=30, size=90).tolist()
# Inject anomalies
historical_durations[85] = 900   # 3x normal — timeout
historical_durations[86] = 50    # too fast — likely data missing
historical_durations[87] = 750   # 2.5x normal — slow

run_data = pd.DataFrame({
    "run_id":       [f"RUN_{i:04d}" for i in range(len(historical_durations))],
    "dag_id":       "etl_daily_orders",
    "duration_sec": historical_durations,
    "rows_written":  np.random.randint(90000, 110000, size=len(historical_durations)).tolist(),
})
# Inject row count anomaly
run_data.at[88, "rows_written"] = 0  # zero rows — data missing!

con = duckdb.connect(":memory:")
con.register("etl_runs", run_data)

# Anomaly detection using Z-score and IQR
alerts = con.execute("""
    WITH stats AS (
        SELECT
            dag_id,
            AVG(duration_sec)                    AS mean_dur,
            STDDEV(duration_sec)                 AS std_dur,
            PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY duration_sec) AS q1,
            PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY duration_sec) AS q3,
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY duration_sec) AS p95,
            AVG(rows_written)                    AS mean_rows,
            STDDEV(rows_written)                 AS std_rows
        FROM etl_runs
        WHERE run_id < 'RUN_0085'  -- use historical data to build baseline
        GROUP BY dag_id
    ),
    recent AS (
        SELECT * FROM etl_runs WHERE run_id >= 'RUN_0085'
    )
    SELECT
        r.run_id,
        r.dag_id,
        r.duration_sec,
        r.rows_written,
        ROUND((r.duration_sec - s.mean_dur) / s.std_dur, 2) AS zscore_duration,
        ROUND(s.q3 + 1.5 * (s.q3 - s.q1), 0)               AS iqr_upper_bound,
        CASE
            WHEN ABS((r.duration_sec - s.mean_dur) / s.std_dur) > 3
              THEN 'DURATION_ZSCORE_ANOMALY'
            WHEN r.duration_sec > s.q3 + 1.5 * (s.q3 - s.q1)
              THEN 'DURATION_IQR_ANOMALY'
            WHEN r.duration_sec > s.p95 * 1.5
              THEN 'DURATION_SLA_BREACH'
            WHEN r.rows_written = 0
              THEN 'ZERO_ROWS_WRITTEN'
            ELSE NULL
        END AS anomaly_type
    FROM recent r
    CROSS JOIN stats s ON r.dag_id = s.dag_id
    WHERE anomaly_type IS NOT NULL
""").fetchdf()

print("=== ETL Anomaly Detection Results ===")
if len(alerts) > 0:
    print(alerts.to_string(index=False))
    print(f"\nTotal anomalies detected: {len(alerts)}")
else:
    print("No anomalies detected")

# Alert routing simulation
print("\n=== Alert Routing ===")
for _, alert in alerts.iterrows():
    if alert["anomaly_type"] == "ZERO_ROWS_WRITTEN":
        priority, channel = "P1", "PagerDuty"
    elif "SLA" in alert["anomaly_type"]:
        priority, channel = "P2", "Slack #data-alerts"
    else:
        priority, channel = "P3", "Email"
    print(f"  [{priority}] {alert['run_id']} → {channel}: {alert['anomaly_type']}")

---
## 5. 设计 Multi-tenant 数仓隔离

### 5.1 需求澄清

- 租户数：50 个企业客户（SaaS 产品的各个企业用户）
- 隔离需求：租户 A 的数据不能被租户 B 读取
- 成本隔离：每个租户独立计费
- 查询配额：防止某租户影响其他租户（Noisy Neighbor）

### 5.2 隔离模型对比

```
隔离程度（强 → 弱）：

┌─────────────────────────────────────────────────────────────┐
│  方案 1: 独立集群 (Separate Clusters)                       │
│  每个租户独立 Redshift / BigQuery 项目 / Snowflake Account  │
│  ✓ 最强隔离   ✗ 成本最高   ✗ 管理复杂                      │
├─────────────────────────────────────────────────────────────┤
│  方案 2: 独立数据库 (Separate Databases)                    │
│  同一集群，每个租户独立 database/schema                     │
│  ✓ 强隔离     ✓ 中等成本   ✓ 中等复杂度                    │
├─────────────────────────────────────────────────────────────┤
│  方案 3: 独立 Schema (Separate Schemas)                     │
│  同一 database，schema = tenant_id                         │
│  ✓ 中等隔离   ✓ 较低成本   ✓ 中等复杂度                    │
├─────────────────────────────────────────────────────────────┤
│  方案 4: 行级安全 (Row-Level Security)                      │
│  共享表 + WHERE tenant_id = current_user_tenant()          │
│  ✓ 最低成本   ✗ 最弱隔离   ✗ 有 RLS 漏洞风险               │
└─────────────────────────────────────────────────────────────┘
```

### 5.3 推荐方案：Separate Schema + RBAC + 查询配额

```
Snowflake 架构示例：

ACCOUNT
  └── DATABASE: saas_dwh
        ├── SCHEMA: tenant_001    ← Tenant A 的所有表
        │     ├── TABLE: orders
        │     ├── TABLE: users
        │     └── TABLE: events
        ├── SCHEMA: tenant_002    ← Tenant B 的所有表
        │     └── ...
        └── SCHEMA: shared        ← 公共维度表（产品目录等）

RBAC:
  ROLE: tenant_001_admin  → OWNERSHIP on SCHEMA tenant_001
  ROLE: tenant_001_reader → SELECT on SCHEMA tenant_001
  USER: tenant_001_svc    → ROLE tenant_001_reader

查询配额 (Resource Monitor):
  CREATE RESOURCE MONITOR tenant_001_monitor
    CREDIT_QUOTA = 100   -- per month
    NOTIFY_AT 80         -- alert at 80%
    SUSPEND_AT 100;      -- suspend at 100%
```

### 5.4 BigQuery IAM 对比

```
BigQuery 方案（Dataset = Schema）：

PROJECT: saas-dwh-prod
  ├── DATASET: tenant_001
  │     IAM: serviceAccount:tenant001@... → roles/bigquery.dataViewer
  ├── DATASET: tenant_002
  │     IAM: serviceAccount:tenant002@... → roles/bigquery.dataViewer
  └── DATASET: shared
        IAM: allAuthenticatedUsers → roles/bigquery.dataViewer

列级别安全（Column-Level Security）：
  -- 敏感列（如 PII）打标签，限制特定角色才能查看
  CREATE OR REPLACE TABLE tenant_001.users AS ...
    OPTIONS (description = 'Contains PII data');
```

### 5.5 成本分摊设计

```sql
-- 查询成本追踪（Snowflake）
SELECT
    query_tag,          -- set by application: 'tenant_id=001'
    SUM(credits_used),
    COUNT(*) AS query_count
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD(month, -1, CURRENT_DATE)
GROUP BY query_tag
ORDER BY SUM(credits_used) DESC;
```

In [ ]:
# Simulate multi-tenant isolation with Row-Level Security using DuckDB
import duckdb
import pandas as pd

con = duckdb.connect(":memory:")

# Shared table with tenant_id (Row-Level Security pattern)
con.execute("""
CREATE TABLE orders_shared (
    order_id    VARCHAR(20),
    tenant_id   VARCHAR(20) NOT NULL,  -- multi-tenant discriminator
    customer_id VARCHAR(20),
    amount      DECIMAL(10,2),
    order_date  DATE
);

INSERT INTO orders_shared VALUES
    ('ORD001', 'tenant_001', 'C001', 500.00, '2024-01-01'),
    ('ORD002', 'tenant_001', 'C002', 300.00, '2024-01-02'),
    ('ORD003', 'tenant_002', 'C003', 800.00, '2024-01-01'),
    ('ORD004', 'tenant_002', 'C004', 200.00, '2024-01-03'),
    ('ORD005', 'tenant_003', 'C005', 950.00, '2024-01-01');

-- RLS View: simulate what tenant_001 would see
-- In production: CREATE ROW ACCESS POLICY in Snowflake / BigQuery
CREATE VIEW orders_tenant_001 AS
    SELECT order_id, customer_id, amount, order_date
    FROM orders_shared
    WHERE tenant_id = 'tenant_001';  -- enforced at view level
""")

print("=== Shared table (admin view — all tenants) ===")
print(con.execute("SELECT * FROM orders_shared").fetchdf().to_string(index=False))

print("\n=== Tenant 001 view (RLS applied — only their data) ===")
print(con.execute("SELECT * FROM orders_tenant_001").fetchdf().to_string(index=False))

# Query quota simulation: track queries per tenant
print("\n=== Query Quota Tracking ===")
quota_sim = pd.DataFrame([
    {"tenant_id": "tenant_001", "queries_this_month": 8500,  "credit_limit": 10000, "credits_used": 82.5},
    {"tenant_id": "tenant_002", "queries_this_month": 3200,  "credit_limit": 5000,  "credits_used": 30.1},
    {"tenant_id": "tenant_003", "queries_this_month": 12100, "credit_limit": 10000, "credits_used": 105.2},
])
quota_sim["pct_used"] = (quota_sim["credits_used"] / quota_sim["credit_limit"] * 100).round(1)
quota_sim["status"] = quota_sim["pct_used"].apply(
    lambda x: "SUSPENDED" if x >= 100 else ("WARNING" if x >= 80 else "OK")
)
print(quota_sim.to_string(index=False))

# Cost allocation report
print("\n=== Multi-tenant Isolation Strategy Comparison ===")
strategies = pd.DataFrame([
    {"Strategy": "Separate Clusters",  "Isolation": "Strongest", "Cost": "Highest",  "Complexity": "High",   "Best for": "Compliance/Enterprise"},
    {"Strategy": "Separate Databases", "Isolation": "Strong",    "Cost": "Medium",   "Complexity": "Medium", "Best for": "50-200 tenants"},
    {"Strategy": "Separate Schemas",   "Isolation": "Medium",    "Cost": "Low",      "Complexity": "Medium", "Best for": "100-1000 tenants"},
    {"Strategy": "Row-Level Security", "Isolation": "Weakest",   "Cost": "Lowest",   "Complexity": "Low",    "Best for": "1000+ tenants, low sensitivity"},
])
print(strategies.to_string(index=False))

---
## 复习要点

### 系统设计答题框架
每题必须覆盖：**需求澄清 → 架构图 → 组件设计 → 数据模型 → 规模估算 → 权衡**

### Clickstream 系统
- Kafka 按 user_id 分区 → 同用户事件有序，Session 重建不需跨 partition
- Flink Watermark 处理乱序（允许 N 分钟延迟），晚到数据单独处理
- 去重：event_id + RocksDB 状态（精确）或 HyperLogLog（近似）

### Lambda vs Kappa
- Lambda：批（精确）+ 流（近实时），两套代码，适合需要精确历史计算
- Kappa：只有流，历史 = 重放 Kafka，适合简单聚合、团队小
- 实时 Dashboard 常用 Druid/Pinot 作 Serving Layer

### Feature Store
- 离线（训练）+ 在线（推理）双存储
- **Point-in-Time Correct Join** 是核心，防止数据泄露
- 特征注册表（Registry）管理元数据、血缘、版本
- Feast（开源）vs Tecton（商业）

### ETL 监控
- 指标：运行时长、行数、SLA 截止时间、错误率
- 检测：规则（P95×1.5）+ 统计（Z-score/IQR）+ ML（季节性）
- 告警：去重、分级、升级、静默窗口
- 自动修复只对可安全重试的故障，数据质量问题必须人工判断

### Multi-tenant
- 隔离强度：独立集群 > 独立数据库 > 独立 Schema > RLS
- 成本与隔离往往反比：越强越贵
- Snowflake：RBAC + Resource Monitor；BigQuery：IAM + Column-Level Security
- 50 租户推荐：Separate Schema + RBAC + 查询配额

---
## 练习

以下为完整系统设计练习题，建议每题限时 30-45 分钟作答。

### 练习 1 — 设计电商实时库存系统

**题目**：设计一个支持 1000 个 SKU、每秒 5000 次库存更新的实时库存管理系统。
要求：
- 读取延迟 < 50ms（库存查询）
- 写入延迟 < 200ms（扣减库存）
- 不允许超卖
- 支持库存历史审计（每次变更可追溯）
- 当库存低于阈值时自动告警

**请按 6 步模板作答**：
1. 需求澄清（写出你要问的 3-5 个问题及假设答案）
2. 高层架构（ASCII 图）
3. 关键组件设计（如何防止超卖？Redis WATCH？分布式锁？）
4. 数据模型（库存表 + 变更日志表 DDL）
5. 规模估算
6. 权衡讨论（强一致性 vs 高可用的取舍）

**评分标准**：
- 防超卖必须提到：Redis 原子操作（DECRBY + 检查）或 DB 乐观锁（版本号）
- 架构中有明确的读写分离
- 权衡讨论中提到 CAP 定理（CP vs AP 选择）

**你的回答**：

```
1. 需求澄清：
   ...

2. 高层架构：
   ...

3. 关键组件设计：
   ...

4. 数据模型：
   ...

5. 规模估算：
   ...

6. 权衡讨论：
   ...
```

### 练习 2 — 设计数据血缘追踪系统

**题目**：你的公司有 5000 张数仓表，500 个 dbt 模型，200 个 Airflow DAG。
设计一个数据血缘追踪系统，要求：
- 能查询「某张表的上游来源」和「某张表的下游影响」
- 当某个源头表变更 schema 时，能快速找到所有受影响的下游
- 支持列级别血缘（哪列来自哪列）
- 血缘图可视化（DAG 展示）

**问题**：
1. 数据血缘的采集方式有哪些？各有什么优缺点？（至少 3 种）
2. 血缘数据如何存储？图数据库（Neo4j）vs 关系数据库的对比
3. 用 SQL（或 Python）实现：给定一张目标表，找出所有直接和间接上游表（BFS/DFS）
4. 如何处理跨系统血缘（Kafka topic → Spark job → Delta table → dbt model → Tableau dashboard）？

**评分标准**：
- 血缘采集：SQL 解析（sqlglot/sqlparse）、运行时 Hook/Listener、元数据 API
- 图遍历：递归 CTE 或 BFS 实现
- 提到 OpenLineage / Marquez 等标准

In [ ]:
# Exercise 2: Implement lineage graph traversal (BFS) using recursive CTE
import duckdb

con = duckdb.connect(":memory:")

# Lineage edges: (source_table, target_table)
con.execute("""
CREATE TABLE lineage_edges (
    source_table  VARCHAR(100),
    target_table  VARCHAR(100),
    edge_type     VARCHAR(50)   -- 'transform', 'copy', 'aggregate'
);
INSERT INTO lineage_edges VALUES
    ('raw.orders',         'bronze.orders',       'copy'),
    ('raw.customers',      'bronze.customers',    'copy'),
    ('bronze.orders',      'silver.orders',       'transform'),
    ('bronze.customers',   'silver.customers',    'transform'),
    ('silver.orders',      'gold.daily_revenue',  'aggregate'),
    ('silver.customers',   'gold.customer_360',   'transform'),
    ('silver.orders',      'gold.customer_360',   'transform'),
    ('gold.daily_revenue', 'mart.exec_dashboard', 'copy'),
    ('gold.customer_360',  'mart.exec_dashboard', 'copy');
""")

# TODO: Implement upstream lineage traversal using recursive CTE
# Find all upstream tables for 'mart.exec_dashboard'
upstream = con.execute("""
    WITH RECURSIVE upstream(table_name, depth, path) AS (
        -- Base case: start from target table
        SELECT 'mart.exec_dashboard', 0, ['mart.exec_dashboard']
        UNION ALL
        -- Recursive: find sources of current tables
        SELECT e.source_table, u.depth + 1, list_append(u.path, e.source_table)
        FROM upstream u
        JOIN lineage_edges e ON u.table_name = e.target_table
        WHERE u.depth < 10  -- prevent infinite loops
          AND NOT list_contains(u.path, e.source_table)  -- cycle detection
    )
    SELECT DISTINCT table_name, depth
    FROM upstream
    WHERE table_name != 'mart.exec_dashboard'
    ORDER BY depth, table_name
""").fetchdf()

print("=== All upstream tables of 'mart.exec_dashboard' ===")
print(upstream.to_string(index=False))

# Impact analysis: if 'raw.orders' changes, what downstream is affected?
downstream = con.execute("""
    WITH RECURSIVE downstream(table_name, depth) AS (
        SELECT 'raw.orders', 0
        UNION ALL
        SELECT e.target_table, d.depth + 1
        FROM downstream d
        JOIN lineage_edges e ON d.table_name = e.source_table
        WHERE d.depth < 10
    )
    SELECT DISTINCT table_name, depth
    FROM downstream
    WHERE table_name != 'raw.orders'
    ORDER BY depth, table_name
""").fetchdf()

print("\n=== Impact analysis: downstream of 'raw.orders' ===")
print(downstream.to_string(index=False))

### 练习 3 — Feature Store 设计问答

**场景**：你在一家推荐系统公司，ML 团队反映：
1. 每次训练新模型都要重新计算相同的特征，浪费 3-4 小时
2. 线上推理时特征计算逻辑和离线训练时不一致（Training-Serving Skew）
3. 无法知道某个特征是谁创建的、数据来源是什么

**问题**：
1. 这 3 个问题分别对应 Feature Store 的哪个核心能力？
2. 什么是 Training-Serving Skew？举出 2 个具体的数据工程原因
3. 设计 `feature_registry` 表，需要记录哪些元数据？
4. 用 Python 实现一个简化版的特征查找函数 `get_features(entity_ids, feature_names, as_of_timestamp)`

**评分标准**：
- 正确识别 Feature Store 的三大价值：特征复用、一致性保证、可发现性
- Training-Serving Skew 的具体原因（如离线用 Spark 算、在线用 Python 重写，逻辑不同）
- `as_of_timestamp` 实现必须涉及 Point-in-Time Correct

In [ ]:
# Exercise 3: Implement simplified feature store lookup
import duckdb
import pandas as pd
from datetime import datetime

con = duckdb.connect(":memory:")

# Setup feature tables
con.execute("""
CREATE TABLE feature_user_stats (
    entity_id         VARCHAR(20),
    feature_timestamp TIMESTAMP,
    purchase_count_7d INTEGER,
    avg_order_value   DOUBLE,
    days_since_login  INTEGER
);
INSERT INTO feature_user_stats VALUES
    ('U001', '2024-01-01 00:00:00', 3,  120.0, 1),
    ('U001', '2024-01-05 00:00:00', 5,  135.0, 5),
    ('U001', '2024-01-10 00:00:00', 8,  142.0, 0),
    ('U002', '2024-01-01 00:00:00', 1,   80.0, 10),
    ('U002', '2024-01-08 00:00:00', 2,   90.0, 7);
""")

def get_features(
    entity_ids: list[str],
    feature_names: list[str],
    as_of_timestamp: datetime
) -> pd.DataFrame:
    """
    Retrieve features for given entity IDs as of a specific timestamp.
    Implements Point-in-Time Correct lookup.
    """
    # TODO: implement PIT-correct feature lookup
    # Hint: for each entity_id, find the row with
    # max(feature_timestamp) where feature_timestamp <= as_of_timestamp

    # Build column selection from feature_names
    cols = ", ".join(["entity_id"] + [f for f in feature_names if f in
                                       ["purchase_count_7d", "avg_order_value", "days_since_login"]])
    ids_str = ", ".join([f"'{eid}'" for eid in entity_ids])

    result = con.execute(f"""
        SELECT {cols}, feature_timestamp AS feature_as_of
        FROM feature_user_stats
        WHERE entity_id IN ({ids_str})
          AND feature_timestamp <= '{as_of_timestamp}'
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY entity_id
            ORDER BY feature_timestamp DESC
        ) = 1
        ORDER BY entity_id
    """).fetchdf()
    return result

# Test: get features as of Jan 7 (should use Jan 5 data for U001, Jan 1 for U002)
features = get_features(
    entity_ids=["U001", "U002"],
    feature_names=["purchase_count_7d", "avg_order_value"],
    as_of_timestamp=datetime(2024, 1, 7)
)
print("=== Feature lookup as of 2024-01-07 ===")
print(features.to_string(index=False))

### 练习 4 — ETL 监控：设计 SLA 告警系统

**场景**：你的 Gold Layer 有 20 张关键表，每天需要在 07:00 前完成更新（SLA）。
现在是 06:45，有 5 张表还没完成，你需要判断哪些会违反 SLA。

**问题**：
1. 用 Python/SQL 实现：根据历史运行时间，预测当前未完成的任务是否会超过 07:00
2. 设计告警升级策略：06:30 还未完成 → Slack，06:50 还未完成 → PagerDuty，07:00 超时 → PagerDuty P1
3. 如果一个上游 DAG 失败，如何自动找到所有依赖它的下游 DAG 并预先告警？
4. 误报率过高（每天 20+ 条误报）会有什么后果？如何降低误报率？

**评分标准**：
- SLA 预测需用历史 P95 或均值+方差，不能只用上一次运行时间
- 告警升级策略需要说明如何避免重复通知
- 误报后果：告警疲劳（Alert Fatigue），工程师开始忽略告警

In [ ]:
# Exercise 4: SLA prediction based on historical run times
import pandas as pd
import numpy as np
import duckdb
from datetime import datetime, timedelta

con = duckdb.connect(":memory:")
np.random.seed(42)

# Historical run data: 30 days of job durations
jobs = ["job_orders", "job_customers", "job_revenue", "job_inventory", "job_sessions"]
base_durations = {"job_orders": 25, "job_customers": 15, "job_revenue": 45,
                  "job_inventory": 20, "job_sessions": 35}  # mean minutes

historical = []
for job in jobs:
    mean = base_durations[job]
    for day in range(30):
        historical.append({
            "job_id": job,
            "run_date": f"2024-01-{day+1:02d}",
            "start_time": f"05:{np.random.randint(0,59):02d}:00",
            "duration_min": max(5, int(np.random.normal(mean, mean*0.15)))
        })

hist_df = pd.DataFrame(historical)
con.register("job_history", hist_df)

# Current run status (today at 06:45)
current_time = datetime(2024, 2, 1, 6, 45)
current_runs = pd.DataFrame([
    {"job_id": "job_orders",    "start_time": "06:10:00", "status": "running"},
    {"job_id": "job_customers", "start_time": "06:30:00", "status": "running"},
    {"job_id": "job_revenue",   "start_time": "05:50:00", "status": "running"},
    {"job_id": "job_inventory", "start_time": "06:25:00", "status": "running"},
    {"job_id": "job_sessions",  "start_time": "05:40:00", "status": "running"},
])
con.register("current_runs", current_runs)

# TODO: Predict which jobs will breach the 07:00 SLA
# For each running job: start_time + P95_historical_duration → predicted_end_time
# If predicted_end_time > 07:00 → SLA BREACH RISK

sla_forecast = con.execute("""
    WITH historical_stats AS (
        SELECT
            job_id,
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY duration_min) AS p95_duration,
            AVG(duration_min) AS avg_duration
        FROM job_history
        GROUP BY job_id
    ),
    with_forecast AS (
        SELECT
            r.job_id,
            r.start_time,
            r.status,
            s.p95_duration,
            -- Elapsed minutes since start (current_time = 06:45)
            DATEDIFF('minute',
                CAST('06:45:00' AS TIME) - CAST(r.start_time AS TIME),
                CAST('06:45:00' AS TIME)) AS elapsed_min,
            -- Predicted completion = start + P95 duration
            -- Expressed as minutes past midnight
            (EXTRACT(HOUR FROM CAST(r.start_time AS TIME)) * 60
             + EXTRACT(MINUTE FROM CAST(r.start_time AS TIME))
             + s.p95_duration)           AS predicted_end_min
        FROM current_runs r
        JOIN historical_stats s USING (job_id)
    )
    SELECT
        job_id,
        start_time,
        ROUND(p95_duration, 1)  AS p95_min,
        ROUND(predicted_end_min, 0)  AS pred_end_min_past_midnight,
        -- 07:00 = 420 minutes past midnight
        CASE WHEN predicted_end_min > 420 THEN 'SLA_BREACH_RISK' ELSE 'ON_TRACK' END AS sla_status,
        ROUND(predicted_end_min - 420, 0) AS minutes_over_sla
    FROM with_forecast
    ORDER BY predicted_end_min DESC
""").fetchdf()

print("=== SLA Forecast at 06:45 (SLA deadline: 07:00 = 420 min) ===")
print(sla_forecast.to_string(index=False))

### 练习 5 — Multi-tenant 隔离设计

**场景**：你们公司有 200 个企业客户，计划迁移到新的数仓架构。
合规团队要求：金融行业客户（30 个）必须完全隔离，其他客户可以共享。

**问题**：
1. 为金融客户和普通客户分别推荐隔离方案，并给出理由
2. 如何设计 DDL 让同一个 dbt 模型（如 `orders`）能同时支持 200 个租户的 schema？
3. 某个租户的查询突然消耗大量计算资源影响了其他租户，你如何检测和处理这个 Noisy Neighbor 问题？
4. 设计一个成本分摊报告 SQL，按租户汇总本月的 Snowflake Credits 消耗

**评分标准**：
- 金融客户：独立账号/集群或独立数据库
- dbt 方案：`generate_schema_name` macro 动态生成 schema
- Noisy Neighbor：Resource Monitor + Query Timeout + 并发限制

In [ ]:
# Exercise 5: Multi-tenant cost allocation report
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect(":memory:")
np.random.seed(0)

# Simulate Snowflake query_history with tenant tags
tenants = [f"tenant_{i:03d}" for i in range(1, 11)]
tenant_types = {t: "finance" if int(t.split("_")[1]) <= 3 else "standard" for t in tenants}

query_log = []
for _ in range(200):
    tenant = np.random.choice(tenants)
    credits = np.random.exponential(scale=0.5 if tenant_types[tenant] == "finance" else 0.2)
    query_log.append({
        "query_id": f"Q{np.random.randint(100000,999999)}",
        "tenant_id": tenant,
        "tenant_type": tenant_types[tenant],
        "query_date": f"2024-01-{np.random.randint(1,32):02d}",
        "credits_used": round(credits, 4),
        "duration_sec": int(credits * 100 + np.random.randint(1, 30))
    })

log_df = pd.DataFrame(query_log)
con.register("query_log", log_df)

# Cost allocation report
cost_report = con.execute("""
    SELECT
        tenant_id,
        tenant_type,
        COUNT(*)                            AS query_count,
        ROUND(SUM(credits_used), 2)         AS total_credits,
        ROUND(AVG(credits_used), 4)         AS avg_credits_per_query,
        ROUND(MAX(credits_used), 4)         AS max_single_query,
        ROUND(AVG(duration_sec), 1)         AS avg_duration_sec,
        -- Identify noisy neighbors: top consumer
        RANK() OVER (ORDER BY SUM(credits_used) DESC)  AS cost_rank
    FROM query_log
    GROUP BY tenant_id, tenant_type
    ORDER BY total_credits DESC
""").fetchdf()

print("=== Monthly Cost Allocation Report ===")
print(cost_report.to_string(index=False))

# Noisy neighbor detection: query consuming > 2x average
print("\n=== Noisy Neighbor Detection (queries > 2x tenant average) ===")
noisy = con.execute("""
    WITH tenant_avg AS (
        SELECT tenant_id, AVG(credits_used) AS avg_credits
        FROM query_log GROUP BY tenant_id
    )
    SELECT q.query_id, q.tenant_id, q.credits_used,
           ROUND(t.avg_credits, 4) AS tenant_avg,
           ROUND(q.credits_used / t.avg_credits, 1) AS ratio
    FROM query_log q JOIN tenant_avg t USING (tenant_id)
    WHERE q.credits_used > t.avg_credits * 2
    ORDER BY ratio DESC
    LIMIT 5
""").fetchdf()
print(noisy.to_string(index=False) if len(noisy) > 0 else "No noisy queries detected")

print("\n=== dbt Schema Generation Macro (Snowflake) ===")
print("""
-- dbt macros/generate_schema_name.sql
{% macro generate_schema_name(custom_schema_name, node) -%}
    {%- set tenant_id = var('tenant_id', 'shared') -%}
    {%- if custom_schema_name -%}
        {{ tenant_id }}_{{ custom_schema_name | trim }}
    {%- else -%}
        {{ tenant_id }}
    {%- endif -%}
{%- endmacro %}

-- Usage: dbt run --vars '{tenant_id: tenant_001}'
-- Result: models land in schema 'tenant_001' or 'tenant_001_staging'
""")